Transformer block from scratch

Build the core components: Multi-Head Attention, FFN, and LayerNorm

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# scaled dot product attention
def attention(Q,K,V,mask=None):
  """
    Q, K, V: [batch, heads, seq_len, d_k]
    mask: [1, 1, seq_len, seq_len] boolean - True=attend, False=ignore.
          For causal (autoregressive) masking, use torch.tril(torch.ones(seq, seq)).
    Returns: [batch, heads, seq_len, d_k]
  """
  d_k = Q.size(-1)

  # compute attention scores
  scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(d_k)

  # apply mask (for causal attention)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, float('inf'))


  # softmax -> weights sum to 1
  weights = F.softmax(scores, dim=-1)

  return torch.matmul(weights,V)

# Multi Head Attention

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model = 512, n_heads = 8):
    super().__init__()
    self.n_heads = n_heads
    self.d_k = d_model // n_heads

    # Projections for Q, K, V
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def forward(self, x, mask=None):
    batch, seq_len, _ = x.shape

    # Project and reshape: [batch, seq, d_model] → [batch, heads, seq, d_k]
    Q = self.W_q(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1,2)
    K = self.W_k(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
    V = self.W_v(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

    # apply attention
    attn_out = attention(Q,K,V,mask)

    attn_out = attn_out.transpose(1,2).contiguous().view(batch, seq_len, -1)

    return self.W_o(attn_out)


# transformer block

class TransformerBlock(nn.Module):
  def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
    super().__init__()
    self.attention = MultiHeadAttention(d_model, n_heads)
    self.norm1 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
    )
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    # Pre-LN: normalize BEFORE sublayer (used by GPT, Llama)
    # More stable training than Post-LN (original paper)
    # Sub-layer 1: Norm → Multi-Head Attention → Residual
    attn_out = self.attention(self.norm1(x), mask)
    x = x + self.dropout(attn_out)

    # sub layer 2:  norm -> ffn -> residual
    ffn_out = self.ffn(self.norm2(x))
    x = x + self.dropout(ffn_out)
    return x


# testing
block = TransformerBlock(d_model=512,n_heads=8)
x = torch.randn(2,10,512) #  [batch=2, seq_len=10, d_model=512]

# without mask (bidirectional like bert encoder)
output = block(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Params: {sum(p.numel() for p in block.parameters()):,}")

# with causal mask
seq_len = x.size(1)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0) # [1,1,seq,seq]
output_causal = block(x, mask=causal_mask)
print(f"Causal output: {output_causal.shape} (each position sees only past tokens)")

Input: torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512])
Params: 3,152,384
Causal output: torch.Size([2, 10, 512]) (each position sees only past tokens)
